In [1]:
import time
from datetime import datetime
from google_play_scraper import Sort, reviews
import pandas as pd

# Target Ethiopian banking applications (Verified Package IDs)
TARGET_BANKS = {
    "Commercial Bank of Ethiopia (CBE)": "com.combanketh.mobilebanking",
    "Bank of Abyssinia (BOA)": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Parameters
MIN_REVIEWS_PER_BANK = 400
BATCH_SIZE = 200  # Maximum batch limit permitted by the API

def scrape_bank_reviews(app_name, app_id, target_count):
    print(f"🔄 Starting data collection for: {app_name} ({app_id})...")
    collected_reviews = []
    continuation_token = None

    while len(collected_reviews) < target_count:
        try:
            # Initial pagination block execution
            if continuation_token is None:
                batch_results, continuation_token = reviews(
                    app_id,
                    lang="en",       # Primary language for data analysis
                    country="et",    # Target region specific to Ethiopia 
                    sort=Sort.NEWEST, # Grabs most recent downwards
                    count=BATCH_SIZE
                )
            else:
                # Sequential pagination leveraging the continuation token
                batch_results, continuation_token = reviews(
                    app_id, 
                    continuation_token=continuation_token
                )

            # Safety fallback if the API serves an empty dataset array
            if not batch_results:
                print(f"⚠️ App stream dry or end of available database reached for {app_name}.")
                break

            for rev in batch_results:
                review_data = {
                    "review text": rev.get("content"),
                    "rating (1–5)": rev.get("score"),
                    "review date": rev.get("at"),
                    "bank / app name": app_name,
                    "source": "Google Play"
                }
                collected_reviews.append(review_data)

            print(f"   Collected {len(collected_reviews)} reviews so far...")

            # Break safely if Google Play stops supplying subsequent pagination tokens
            if not continuation_token:
                print(f"ℹ️ Reached terminal token checkpoint for {app_name}.")
                break

            # Rate limit/anti-bot cooldown strategy
            time.sleep(2.5)

        except Exception as e:
            print(f"❌ Error extracting data for {app_name}: {e}")
            break

    # Hard truncate to prevent spilling past target threshold
    return collected_reviews[:target_count]

# Main Pipeline Runner
all_scraped_data = []

for bank_name, package_id in TARGET_BANKS.items():
    bank_data = scrape_bank_reviews(bank_name, package_id, MIN_REVIEWS_PER_BANK)
    all_scraped_data.extend(bank_data)
    print(f"✅ Completed {bank_name}. Records collected: {len(bank_data)}\n")
    time.sleep(3.5) # Inter-app network buffer pause

# DataFrame Compilation
df = pd.DataFrame(all_scraped_data)

# Execution Meta-Report and Boundary Documentation
print("--- DATASET SCRAPING SUMMARY REPORT ---")
print(f"Grand Total Records Captured: {len(df)}")
print("\nBreakdown Volume by Subject Asset:")
print(df["bank / app name"].value_counts())

print("\nTemporal Boundaries Checked:")
for bank in TARGET_BANKS.keys():
    bank_slice = df[df["bank / app name"] == bank]
    if not bank_slice.empty:
        min_date = bank_slice["review date"].min()
        max_date = bank_slice["review date"].max()
        print(f" - {bank} Date Range: From {min_date} to {max_date}")
    else:
        print(f" - ⚠️ {bank}: No data scraped. Check internet parameters or App ID status.")

# Save output to disk
df.to_csv("ethiopian_banks_reviews.csv", index=False, encoding="utf-8")
print("\n💾 Dataset written to disk -> 'ethiopian_banks_reviews.csv'")

🔄 Starting data collection for: Commercial Bank of Ethiopia (CBE) (com.combanketh.mobilebanking)...
   Collected 200 reviews so far...
   Collected 400 reviews so far...
✅ Completed Commercial Bank of Ethiopia (CBE). Records collected: 400

🔄 Starting data collection for: Bank of Abyssinia (BOA) (com.boa.boaMobileBanking)...
   Collected 200 reviews so far...
   Collected 400 reviews so far...
✅ Completed Bank of Abyssinia (BOA). Records collected: 400

🔄 Starting data collection for: Dashen Bank (com.dashen.dashensuperapp)...
   Collected 200 reviews so far...
   Collected 400 reviews so far...
✅ Completed Dashen Bank. Records collected: 400

--- DATASET SCRAPING SUMMARY REPORT ---
Grand Total Records Captured: 1200

Breakdown Volume by Subject Asset:
bank / app name
Commercial Bank of Ethiopia (CBE)    400
Bank of Abyssinia (BOA)              400
Dashen Bank                          400
Name: count, dtype: int64

Temporal Boundaries Checked:
 - Commercial Bank of Ethiopia (CBE) Date 